Project 2 


In [1]:
import os
import sys
from pathlib import Path
# root = Path.cwd().parent
# sys.path.insert(0, str(root))
# if os.path.basename(os.getcwd()) == "Project2":
#     os.chdir("..")
print(os.getcwd())
import part1
import part2

import nltk
import numpy as np
import random
import itertools
import torch
import torch.nn as nn
import pickle
from collections import OrderedDict, Counter

/home/thymon/Documents/UvA/master/nlp/Projects/NLP_MAIN/NLP/Project2


In [2]:
# nltk.download(['stopwords', 'punkt_tab'], force=True)

In [3]:
RESULTS_STORE = "results/stored_results"
os.makedirs(RESULTS_STORE, exist_ok=True)
my_classifiers = OrderedDict()
def save_system(name):
    with open(f"{RESULTS_STORE}/{name}.pkl", "wb") as f:
        pickle.dump(my_classifiers[name], f)

def load_all_systems():
    loaded = OrderedDict()
    for fname in sorted(os.listdir(RESULTS_STORE)):
        if fname.endswith(".pkl"):
            with open(f"{RESULTS_STORE}/{fname}", "rb") as f:
                loaded[fname[:-4]] = pickle.load(f)
    return loaded

In [4]:
my_classifiers = load_all_systems()

In [17]:
def run_system(name, tokenizer, trained_emb_model, enc_type, **model_args):
    my_classifiers[name] = []
    for run, seed in enumerate(my_seeds):
        part2.experiment.cross_validation(
                    data_splits=my_cross_validation_sets[0],
                    outcome_space=stanfordnlpsst2_outcome_space,
                    tokenizer=tokenizer,
                    emb_model=trained_emb_model,
                    enc_type=enc_type,
                    save_to=f"results/{name}/seed-{seed}",
                    **model_args
                    )
    save_system(name)

## Data
SST-2: sentences labelled negative (0) or positive (1). We use the train split for
cross-validation; the official test labels are hidden.

In [6]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")
stanfordnlpsst2_outcome_space = part1.clslib.NamedClasses(['negative', 'positive'])
full_texts = list(dataset["train"]["sentence"]) + list(dataset["validation"]["sentence"])
full_labels = list(dataset["train"]["label"]) + list(dataset["validation"]["label"])

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


## Pre-trained embeddings and tokenizer

In [7]:
# 30 seconds
trained_emb_model_glovevwikigiga50d = part2.clslib.PretrainedEmbeddings(
    "glove.2024.wikigiga.50d.zip",
    50,
    skip_first_line=False,
)


In [8]:
# 3 min run, can crash then just run with restarted kernel (keeps chrashing on my pc)
#trained_emb_model_gloveCC300d = part2.clslib.PretrainedEmbeddings(
#    "glove.42B.300d.zip",
#    300,
#    skip_first_line=False,
#)


In [9]:
# chosen model
trained_emb_model = trained_emb_model_glovevwikigiga50d
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}") 
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1287615, 50)
Size of vocabulary: 1287615
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:the
3:,
4:.
5:of
6:to
7:and
8:in
9:a


In [10]:
tokenizer = part2.clslib.BasicTokenizer(
    lowercase=True, 
    remove_punct=True, 
    remove_stopwords=True,
    language='english',
    vocab=trained_emb_model.vocab
)

unk_id = tokenizer.vocab.default_index

sentences_with_unk = 0
unk_token_occurrences = 0

for text in full_texts:
    ids = tokenizer.ids(text)
    sentences_with_unk += unk_id in ids
    unk_token_occurrences += ids.count(unk_id)

print("Sentences containing at least one UNK:", sentences_with_unk)
print("Total UNK token occurrences:", unk_token_occurrences)

Sentences containing at least one UNK: 503
Total UNK token occurrences: 512


## Comparing embedding vocabulary coverage

Vocabulary coverage was compared by counting how often each embedding vocabulary failed to contain a token in the training sentences. Two counts are reported:

- **Sentences with at least one UNK:** sentences containing one or more out-of-vocabulary tokens.
- **UNK token occurrences:** total out-of-vocabulary tokens across the dataset.

| Embedding model | Sentences with at least one UNK | UNK token occurrences |
|---|---:|---:|
| GloVe 50d (Wiki + Gigaword) | 494 | 503 |
| GloVe 300d (Common Crawl) | 616 | 634 |
| fastText | 973 | 991 |
| Word2Vec | 22,584 | 31,117 |

### Observations

- GloVe 300d (Common Crawl) had the fewest UNK tokens.
- GloVe 50d (wikigiga) also had relatively few UNK tokens.
- fastText had more UNK tokens than either GloVe model.
- Word2Vec had substantially more UNK tokens than the other embeddings.

### Conclusion

Based on vocabulary coverage, both GloVe models will be used for further evaluation. The Glove embeddings are taken from [here](https://github.com/stanfordnlp/GloVe)



In [11]:
def cos_similarity(u, v):  
    """
    Return the cosine similarity (a real number) of the two words in the embedding space of our model.
    """   

    dot = np.dot(u, v)
    norm = np.linalg.norm(u) * np.linalg.norm(v)
    return dot / norm

def topk_ids(v, E, k=10, exclude_ids=()):
    v = np.asarray(v)
    E = np.asarray(E)

    v_norm = np.linalg.norm(v)
    if v_norm == 0:
        raise ValueError("The query word has a zero vector.")

    row_norms = np.linalg.norm(E, axis=1)
    valid = row_norms > 0

    scores = np.full(E.shape[0], -np.inf)
    scores[valid] = (E[valid] @ v) / (row_norms[valid] * v_norm)

    scores[list(exclude_ids)] = -np.inf

    k = min(k, np.isfinite(scores).sum())
    return np.argsort(scores)[-k:][::-1]

In [12]:
words = ['bad', 'horrible', 'neutral', 'amazing', 'terrific', 'good']
for word1 in words:
    topkidx = topk_ids(trained_emb_model.embed_word(word1), trained_emb_model.vectors)
    for idx in topkidx:
        print(f'Top 10 words for {word1}, {trained_emb_model.word(idx)}')

Top 10 words for bad, bad
Top 10 words for bad, worse
Top 10 words for bad, too
Top 10 words for bad, really
Top 10 words for bad, little
Top 10 words for bad, nothing
Top 10 words for bad, unfortunately
Top 10 words for bad, gone
Top 10 words for bad, ?
Top 10 words for bad, things
Top 10 words for horrible, horrible
Top 10 words for horrible, terrible
Top 10 words for horrible, awful
Top 10 words for horrible, dreadful
Top 10 words for horrible, horrific
Top 10 words for horrible, horrendous
Top 10 words for horrible, shocking
Top 10 words for horrible, horrifying
Top 10 words for horrible, tragic
Top 10 words for horrible, scary
Top 10 words for neutral, neutral
Top 10 words for neutral, acceptable
Top 10 words for neutral, sufficiently
Top 10 words for neutral, therefore
Top 10 words for neutral, strictly
Top 10 words for neutral, generally
Top 10 words for neutral, position
Top 10 words for neutral, determined
Top 10 words for neutral, transparent
Top 10 words for neutral, otherwi

In [13]:
u = trained_emb_model.embed_word('neutral') + trained_emb_model.embed_word('not')
topkidx = topk_ids(u, trained_emb_model.vectors)
for idx in topkidx:
    print(f'Top 10 words for u, {trained_emb_model.word(idx)}')

Top 10 words for u, not
Top 10 words for u, be
Top 10 words for u, should
Top 10 words for u, must
Top 10 words for u, if
Top 10 words for u, any
Top 10 words for u, nor
Top 10 words for u, neither
Top 10 words for u, either
Top 10 words for u, longer


In [14]:
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(full_texts, full_labels)), 
            k=num_folds,
            seed=seed
        )
    )

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
print("device:", device)

device: cuda


In [18]:
run_system(
    name="lstm_model",
    tokenizer=tokenizer,
    trained_emb_model=trained_emb_model,
    enc_type="lstm",
    enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.1},
    regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
    opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 17.552 seconds
Dataloader done! Time taken: 0.283 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5092 | train acc: 0.7483 | val loss: 0.4309 | val acc: 0.8012
Epoch 02 | train loss: 0.4502 | train acc: 0.7907 | val loss: 0.4097 | val acc: 0.8126
Epoch 03 | train loss: 0.4284 | train acc: 0.8019 | val loss: 0.3942 | val acc: 0.8245
Epoch 04 | train loss: 0.4156 | train acc: 0.8085 | val loss: 0.3854 | val acc: 0.8286
Epoch 05 | train loss: 0.4046 | train acc: 0.8168 | val loss: 0.3779 | val acc: 0.8336
Epoch 06 | train loss: 0.3967 | train acc: 0.8184 | val loss: 0.3769 | val acc: 0.8382
Epoch 07 | train loss: 0.3885 | train acc: 0.8258 | val loss: 0.3744 | val acc: 0.8338
Epoch 08 | train loss: 0.3819 | train acc: 0.8279 | val loss: 0.3645 | val acc: 0.8415
Epoch 09 | train loss: 0.3758 | train acc: 0.8330 | val loss: 0.3549 | val acc: 0.8460
Epoch 10 | train loss: 0.3711 | train acc: 0.8337 | val loss: 0.3521 | val acc: 0.8501
Epoch 11 | train loss: 0.3639 | train acc: 0.8374 | val loss: 0.3490 | val acc: 0.8514
Epoch 12 | train loss: 0.3627 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5099 | train acc: 0.7464 | val loss: 0.4432 | val acc: 0.7947
Epoch 02 | train loss: 0.4560 | train acc: 0.7829 | val loss: 0.4197 | val acc: 0.8104
Epoch 03 | train loss: 0.4357 | train acc: 0.7948 | val loss: 0.3988 | val acc: 0.8184
Epoch 04 | train loss: 0.4199 | train acc: 0.8072 | val loss: 0.3854 | val acc: 0.8277
Epoch 05 | train loss: 0.4096 | train acc: 0.8134 | val loss: 0.3781 | val acc: 0.8342
Epoch 06 | train loss: 0.3993 | train acc: 0.8188 | val loss: 0.3705 | val acc: 0.8391
Epoch 07 | train loss: 0.3924 | train acc: 0.8228 | val loss: 0.3648 | val acc: 0.8400
Epoch 08 | train loss: 0.3858 | train acc: 0.8257 | val loss: 0.3605 | val acc: 0.8445
Epoch 09 | train loss: 0.3806 | train acc: 0.8291 | val loss: 0.3646 | val acc: 0.8416
Epoch 10 | train loss: 0.3747 | train acc: 0.8324 | val loss: 0.3589 | val acc: 0.8470
Epoch 11 | train loss: 0.3711 | train acc: 0.8353 | val loss: 0.3508 | val acc: 0.8515
Epoch 12 | train loss: 0.3644 | train acc: 

... (2 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5103 | train acc: 0.7473 | val loss: 0.4379 | val acc: 0.7979
Epoch 02 | train loss: 0.4514 | train acc: 0.7858 | val loss: 0.4114 | val acc: 0.8107
Epoch 03 | train loss: 0.4307 | train acc: 0.7980 | val loss: 0.3980 | val acc: 0.8200
Epoch 04 | train loss: 0.4171 | train acc: 0.8057 | val loss: 0.3866 | val acc: 0.8273
Epoch 05 | train loss: 0.4079 | train acc: 0.8127 | val loss: 0.3777 | val acc: 0.8317
Epoch 06 | train loss: 0.3974 | train acc: 0.8185 | val loss: 0.3710 | val acc: 0.8373
Epoch 07 | train loss: 0.3915 | train acc: 0.8224 | val loss: 0.3622 | val acc: 0.8424
Epoch 08 | train loss: 0.3839 | train acc: 0.8254 | val loss: 0.3602 | val acc: 0.8431
Epoch 09 | train loss: 0.3775 | train acc: 0.8307 | val loss: 0.3537 | val acc: 0.8472
Epoch 10 | train loss: 0.3740 | train acc: 0.8321 | val loss: 0.3529 | val acc: 0.8502
Epoch 11 | train loss: 0.3680 | train acc: 0.8359 | val loss: 0.3481 | val acc: 0.8523
Epoch 12 | train loss: 0.3654 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5102 | train acc: 0.7474 | val loss: 0.4389 | val acc: 0.7929
Epoch 02 | train loss: 0.4494 | train acc: 0.7868 | val loss: 0.4176 | val acc: 0.8072
Epoch 03 | train loss: 0.4305 | train acc: 0.7996 | val loss: 0.3982 | val acc: 0.8213
Epoch 04 | train loss: 0.4170 | train acc: 0.8083 | val loss: 0.3888 | val acc: 0.8253
Epoch 05 | train loss: 0.4073 | train acc: 0.8145 | val loss: 0.3782 | val acc: 0.8340
Epoch 06 | train loss: 0.3985 | train acc: 0.8204 | val loss: 0.3730 | val acc: 0.8376
Epoch 07 | train loss: 0.3907 | train acc: 0.8236 | val loss: 0.3723 | val acc: 0.8357
Epoch 08 | train loss: 0.3856 | train acc: 0.8265 | val loss: 0.3629 | val acc: 0.8422
Epoch 09 | train loss: 0.3794 | train acc: 0.8297 | val loss: 0.3592 | val acc: 0.8444
Epoch 10 | train loss: 0.3734 | train acc: 0.8349 | val loss: 0.3559 | val acc: 0.8465
Epoch 11 | train loss: 0.3683 | train acc: 0.8377 | val loss: 0.3521 | val acc: 0.8494
Epoch 12 | train loss: 0.3646 | train acc: 

In [20]:
run_system(
    name="embed-and-pool_model",
    tokenizer=tokenizer,
    trained_emb_model=trained_emb_model,
    enc_type="embed-and-pool",
    enc_kwargs={'freeze_embeddings': True},
    regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
    opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 24.527 seconds
Dataloader done! Time taken: 0.340 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5297 | train acc: 0.7304 | val loss: 0.5009 | val acc: 0.7547
Epoch 02 | train loss: 0.5175 | train acc: 0.7422 | val loss: 0.4970 | val acc: 0.7541
Epoch 03 | train loss: 0.5160 | train acc: 0.7418 | val loss: 0.4937 | val acc: 0.7551
Epoch 04 | train loss: 0.5142 | train acc: 0.7415 | val loss: 0.4911 | val acc: 0.7572
Epoch 05 | train loss: 0.5113 | train acc: 0.7449 | val loss: 0.4875 | val acc: 0.7593
Epoch 06 | train loss: 0.5079 | train acc: 0.7449 | val loss: 0.4871 | val acc: 0.7645
Epoch 07 | train loss: 0.5053 | train acc: 0.7474 | val loss: 0.4814 | val acc: 0.7631
Epoch 08 | train loss: 0.5031 | train acc: 0.7509 | val loss: 0.4815 | val acc: 0.7635
Epoch 09 | train loss: 0.5010 | train acc: 0.7512 | val loss: 0.4784 | val acc: 0.7666
Epoch 10 | train loss: 0.4990 | train acc: 0.7526 | val loss: 0.4725 | val acc: 0.7735
Epoch 11 | train loss: 0.4986 | train acc: 0.7524 | val loss: 0.4729 | val acc: 0.7690
Epoch 12 | train loss: 0.4953 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5282 | train acc: 0.7333 | val loss: 0.5026 | val acc: 0.7547
Epoch 02 | train loss: 0.5203 | train acc: 0.7397 | val loss: 0.4972 | val acc: 0.7564
Epoch 03 | train loss: 0.5174 | train acc: 0.7397 | val loss: 0.4964 | val acc: 0.7560
Epoch 04 | train loss: 0.5147 | train acc: 0.7427 | val loss: 0.4922 | val acc: 0.7577
Epoch 05 | train loss: 0.5103 | train acc: 0.7454 | val loss: 0.4860 | val acc: 0.7649
Epoch 06 | train loss: 0.5073 | train acc: 0.7472 | val loss: 0.4824 | val acc: 0.7657
Epoch 07 | train loss: 0.5043 | train acc: 0.7474 | val loss: 0.4788 | val acc: 0.7694
Epoch 08 | train loss: 0.5018 | train acc: 0.7508 | val loss: 0.4795 | val acc: 0.7706
Epoch 09 | train loss: 0.4997 | train acc: 0.7534 | val loss: 0.4772 | val acc: 0.7713
Epoch 10 | train loss: 0.4987 | train acc: 0.7540 | val loss: 0.4725 | val acc: 0.7724
Epoch 11 | train loss: 0.4987 | train acc: 0.7534 | val loss: 0.4736 | val acc: 0.7732
Epoch 12 | train loss: 0.4953 | train acc: 

... (2 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5295 | train acc: 0.7308 | val loss: 0.5001 | val acc: 0.7506
Epoch 02 | train loss: 0.5177 | train acc: 0.7411 | val loss: 0.4970 | val acc: 0.7502
Epoch 03 | train loss: 0.5147 | train acc: 0.7438 | val loss: 0.4975 | val acc: 0.7550
Epoch 04 | train loss: 0.5115 | train acc: 0.7428 | val loss: 0.4921 | val acc: 0.7555
Epoch 05 | train loss: 0.5081 | train acc: 0.7459 | val loss: 0.4860 | val acc: 0.7616
Epoch 06 | train loss: 0.5058 | train acc: 0.7489 | val loss: 0.4847 | val acc: 0.7627
Epoch 07 | train loss: 0.5028 | train acc: 0.7510 | val loss: 0.4814 | val acc: 0.7655
Epoch 08 | train loss: 0.5019 | train acc: 0.7504 | val loss: 0.4783 | val acc: 0.7657
Epoch 09 | train loss: 0.4968 | train acc: 0.7548 | val loss: 0.4754 | val acc: 0.7715
Epoch 10 | train loss: 0.4971 | train acc: 0.7542 | val loss: 0.4734 | val acc: 0.7728
Epoch 11 | train loss: 0.4938 | train acc: 0.7570 | val loss: 0.4735 | val acc: 0.7757
Epoch 12 | train loss: 0.4932 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5299 | train acc: 0.7322 | val loss: 0.5023 | val acc: 0.7508
Epoch 02 | train loss: 0.5199 | train acc: 0.7418 | val loss: 0.5033 | val acc: 0.7498
Epoch 03 | train loss: 0.5153 | train acc: 0.7435 | val loss: 0.4983 | val acc: 0.7514
Epoch 04 | train loss: 0.5121 | train acc: 0.7433 | val loss: 0.4912 | val acc: 0.7556
Epoch 05 | train loss: 0.5077 | train acc: 0.7455 | val loss: 0.4931 | val acc: 0.7567
Epoch 06 | train loss: 0.5059 | train acc: 0.7470 | val loss: 0.4852 | val acc: 0.7620
Epoch 07 | train loss: 0.5016 | train acc: 0.7522 | val loss: 0.4807 | val acc: 0.7641
Epoch 08 | train loss: 0.4993 | train acc: 0.7532 | val loss: 0.4807 | val acc: 0.7625
Epoch 09 | train loss: 0.5000 | train acc: 0.7542 | val loss: 0.4822 | val acc: 0.7616
Epoch 10 | train loss: 0.4969 | train acc: 0.7539 | val loss: 0.4762 | val acc: 0.7684
Epoch 11 | train loss: 0.4932 | train acc: 0.7588 | val loss: 0.4764 | val acc: 0.7669
Epoch 12 | train loss: 0.4933 | train acc: 